# End-to-End Mode 1 Pipeline

This notebook walks through the complete **PLred Mode 1** data-reduction pipeline — from raw camera files to a calibrated coupling map suitable for image reconstruction. The pipeline processes observations from the SCExAO/FIRST-PL instrument, where a photonic lantern (PL) feeds light into 38 single-mode fibers whose output spectra are recorded on a detector. A fast PSF camera simultaneously records the tip-tilt state, providing the spatial information needed to invert the fiber coupling and reconstruct the on-sky intensity distribution.

```
+--------+-----------------------------+---------------------------+------------------------+
| Step   | Description                 | Input                     | Output                 |
+--------+-----------------------------+---------------------------+------------------------+
| Step 1 | Timestamp matching          | PSFcam FITS + timestamps  | fastcam.h5             |
|        | (PSFcam → PLcam sync)       | PLcam timestamps          | (weighted PSF frames   |
|        |                             |                           |  per PLcam exposure)   |
+--------+-----------------------------+---------------------------+------------------------+
| Step 2 | Ingest PLcam data           | fastcam.h5                | alldata.h5             |
|        | (ROI crop + dark sub)       | PLcam FITS + dark         | (PSF + PLcam frames)   |
+--------+-----------------------------+---------------------------+------------------------+
| Step 3 | ROI viewer cache            | alldata.h5                | roi.h5                 |
|        | (fast pixel time-series)    | ROI bounds                | (transposed cache)     |
+--------+-----------------------------+---------------------------+------------------------+
| Step 4 | Spatial averaging           | alldata.h5                | map.h5                 |
|        | (grid-bin PLcam frames)     | grid params (map_n, xc..) | (avg frame per bin)    |
+--------+-----------------------------+---------------------------+------------------------+
| Step 5 | Trace extraction            | map.h5 + traces.npz       | couplingmap.fits       |
|        | (spectra → coupling map)    | PLcam flat/lamp           | (nbin×nbin×nfib×nwav)  |
+--------+-----------------------------+---------------------------+------------------------+
```

**Prerequisites:** The tutorial data under `tutorials/data/` must be present (fastcam and slowcam sub-directories). All outputs are written to `tutorials/tutorial_output/`. The PLred package must be installed (`pip install -e .` from the repo root).

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import h5py
from astropy.io import fits
from configobj import ConfigObj

# Make sure PLred is importable (adjust if needed)
PLRED_ROOT = Path("__file__").resolve().parent.parent
if str(PLRED_ROOT) not in sys.path:
    sys.path.insert(0, str(PLRED_ROOT))

import PLred.specextract as specextract
from PLred.sort import script_match_timestamps
from PLred.ingest import ingest_from_config_unified
from PLred.average import build_ROI_access_from_config, average_to_h5_from_config, explore_grid

print("Imports OK")

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
TUTORIALS_DIR = Path('/Users/yjkim/Documents/PLred2/PLred-dev/PLred/tutorials')

OUTPUT_DIR = TUTORIALS_DIR / 'tutorial_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OBS_CONFIG = OUTPUT_DIR / 'obs.ini'   # single config for ALL steps
FLAT_FITS  = TUTORIALS_DIR / 'data' / 'slowcam' / 'cropped_firstpl_15:05:17.315924371.fits'
DARK_FITS  = TUTORIALS_DIR / 'data' / 'slowcam' / 'dark.fits'

print('TUTORIALS_DIR :', TUTORIALS_DIR)
print('OUTPUT_DIR    :', OUTPUT_DIR)
print('OBS_CONFIG    :', OBS_CONFIG)


## Step 1: Timestamp Matching → `fastcam.h5`

The PSF camera (Palila, fast camera) records the instantaneous PSF state at ~kilohertz rates.
The PL camera (FIRST-PL, slow camera) integrates for much longer exposures.
The timestamp-matching step assigns each slow PLcam exposure a weighted average of the fast PSF
frames that fell within its integration window.

**Output:** `fastcam.h5` — averaged PSFcam frames, one per matched PLcam exposure.

All five pipeline steps read from a **single `obs.ini`** config file.
The sort-specific sections (`[Fastcam]`, `[Slowcam]`, `[Output]`, `[Options]`) live alongside
the unified sections (`[Ingest]`, `[Average]`, etc.) in the same file —
each CLI reads only the sections it needs and ignores the rest.


In [ ]:
abs_path = str(TUTORIALS_DIR)

cfg = ConfigObj()
cfg.filename = str(OBS_CONFIG)

# ── Sort step (Step 1) ───────────────────────────────────────────────────
cfg['Fastcam'] = {
    'obs_date':   '20240917',
    'start_time': '15:05:10',
    'end_time':   '15:05:11',
    'dark_file':  f'{abs_path}/data/fastcam/dark.fits',
    'dark_start_time': '',
    'dark_end_time':   '',
    'path':       f'{abs_path}/data/fastcam/',
}
cfg['Slowcam'] = {
    'timestamp_dir': f'{abs_path}/data/slowcam/',
    'nbin': '1',
}
cfg['Output'] = {
    'outname':  f'{abs_path}/tutorial_output/',
    'filename': 'fastcam',
}
cfg['Options'] = {
    'verbose':  'False',
    'show_plot': 'False',
    'crop_width': '20',
    'apply_dead_time_correction': 'True',
    'psfcam_is_fast': 'True',
}

# ── Instrument + pipeline steps (Steps 2-5) ──────────────────────────────
cfg['Instrument'] = {
    'nfib': '38',
    'spectral_orientation': 'horizontal',
}
cfg['Ingest'] = {
    'step1_h5':       f'{abs_path}/tutorial_output/fastcam.h5',
    'plcam_dark':     f'{abs_path}/data/slowcam/dark.fits',
    'plcam_data_dir': f'{abs_path}/data/slowcam',
    'plcam_roi':      '0,412,1200,1220',
    'output':         f'{abs_path}/tutorial_output/alldata.h5',
}
cfg['ROIViewer'] = {
    'roi':       '0,412,1200,1220',
    'zarr_path': f'{abs_path}/tutorial_output/roi.h5',
}
cfg['Average'] = {
    'input':       f'{abs_path}/tutorial_output/alldata.h5',
    'map_n':       '5',
    'map_width':   '2',
    'xc':          '18.26',
    'yc':          '20.50',
    'pix2mas':     '16.2',
    'n_bootstrap': '0',
    'output':      f'{abs_path}/tutorial_output/map.h5',
}
cfg['Specextract'] = {
    'input':      f'{abs_path}/tutorial_output/map.h5',
    'extractor':  'trace_box',
    'output':     f'{abs_path}/tutorial_output/couplingmap.fits',
    'plcam_roi':  '0,412,1200,1220',
    'trace_file': f'{abs_path}/tutorial_output/traces.npz',
}

cfg.write()
print('Wrote obs.ini to:', OBS_CONFIG)
print()
with open(OBS_CONFIG) as fh:
    print(fh.read())


In [ ]:
%%time
print("Running timestamp matching ...")
script_match_timestamps(str(OBS_CONFIG))

# Inspect the output
h5_sort = str(OUTPUT_DIR / "fastcam.h5")
with h5py.File(h5_sort, 'r') as f:
    frames     = f['frames'][:]
    timestamps = f['timestamps'][:]
    nstacks    = f['nstacks'][:]

print()
print(f"fastcam.h5 written: {os.path.getsize(h5_sort)/1e6:.1f} MB")
print(f"  n_frames   : {len(frames)}")
print(f"  frames shape: {frames.shape}")
print(f"  timestamps  : [{timestamps.min():.6f}, {timestamps.max():.6f}]")
print(f"  nstacks     : min={nstacks.min()}, max={nstacks.max()}, mean={nstacks.mean():.1f}")

In [ ]:
# Show a mosaic of 6 evenly-spaced PSF frames from fastcam.h5
h5_sort = str(OUTPUT_DIR / "fastcam.h5")
with h5py.File(h5_sort, 'r') as f:
    frames = f['frames'][:]

n = len(frames)
idxs = np.linspace(0, n - 1, 6, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for ax, idx in zip(axes.flat, idxs):
    im = ax.imshow(frames[idx], origin='lower', cmap='inferno')
    ax.set_title(f"Frame {idx}")
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("PSFcam weighted-mean frames (Step 1 output)", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Matched frame count: {n}")

## Step 2: Build `alldata.h5`

The ingest step loads the PLcam FITS files (slow camera), applies the ROI crop defined by `plcam_roi = y0,y1,x0,x1`, subtracts the dark frame, and stores the result as an HDF5 file alongside the PSFcam centroids and peak values derived from `fastcam.h5`. The output `alldata.h5` contains:

- `plcam/frames` — dark-subtracted, ROI-cropped PLcam frames `(N, y1-y0, x1-x0)`
- `psfcam/frames` — dark-subtracted PSFcam frames `(N, crop_h, crop_w)`
- `psfcam/centroids` — PSF centroid `(x, y)` per frame `(N, 2)`
- `psfcam/peaks` — PSF peak value per frame `(N,)`
- `psfcam/timestamps` — timestamps `(N,)`

This step reads `[Instrument]` and `[Ingest]` from the unified config.

In [ ]:
# The unified config was already written in cell 4 above.
# All subsequent steps read from the same obs.ini.
print('Using config:', OBS_CONFIG)
with open(OBS_CONFIG) as fh:
    content = fh.read()
# Show just the pipeline sections (skip sort sections already shown)
in_pipeline = False
for line in content.split('\n'):
    if line.startswith('[Instrument]'):
        in_pipeline = True
    if in_pipeline:
        print(line)


In [ ]:
%%time
print("Running ingest ...")
ingest_from_config_unified(str(OBS_CONFIG))

alldata_h5 = str(OUTPUT_DIR / "alldata.h5")
print()
print(f"alldata.h5 written: {os.path.getsize(alldata_h5)/1e6:.1f} MB")

with h5py.File(alldata_h5, 'r') as f:
    plcam_shape   = f['plcam/frames'].shape
    psfcam_shape  = f['psfcam/frames'].shape
    cent_shape    = f['psfcam/centroids'].shape
    peaks_shape   = f['psfcam/peaks'].shape
    print(f"  plcam/frames shape      : {plcam_shape}")
    print(f"  psfcam/frames shape     : {psfcam_shape}")
    print(f"  psfcam/centroids shape  : {cent_shape}")
    print(f"  psfcam/peaks shape      : {peaks_shape}")
    if 'plcam/dark_subtracted' in f.attrs:
        print(f"  dark subtracted         : {f.attrs['plcam/dark_subtracted']}")
    else:
        print("  (dark subtraction status stored in dataset attrs)")

In [ ]:
alldata_h5 = str(OUTPUT_DIR / "alldata.h5")

with h5py.File(alldata_h5, 'r') as f:
    centroids = f['psfcam/centroids'][:]
    peaks     = f['psfcam/peaks'][:]

cx = centroids[:, 0]
cy = centroids[:, 1]

print(f"Centroid x: mean={cx.mean():.2f}, std={cx.std():.2f}")
print(f"Centroid y: mean={cy.mean():.2f}, std={cy.std():.2f}")
print(f"Peak value: mean={peaks.mean():.1f}, std={peaks.std():.1f}, "
      f"min={peaks.min():.0f}, max={peaks.max():.0f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

sc = ax1.scatter(cx, cy, c=peaks, cmap='viridis', s=10, alpha=0.7)
plt.colorbar(sc, ax=ax1, label='PSF peak (counts)')
ax1.set_xlabel('Centroid x (pixels)')
ax1.set_ylabel('Centroid y (pixels)')
ax1.set_title('PSF centroid positions coloured by peak value')
ax1.set_aspect('equal')

ax2.hist(peaks, bins=30, color='steelblue', edgecolor='k')
ax2.set_xlabel('PSF peak value (counts)')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of PSF peak values')

plt.tight_layout()
plt.show()

## Step 3: ROI Viewer Cache → `roi.h5`

The ROI viewer cache transposes the PLcam frame data so that individual pixel time-series can be read with a small number of I/O operations. Without the cache, reading a single pixel time-series requires loading every frame sequentially; with the cache, only a handful of chunks are needed.

**Interactive HTML viewer:** In a real workflow you would open `PLred/scripts/h5_viewer.html` in a browser (pointing it at `alldata.h5`) to:
- Visualise the centroid scatter plot
- Set peak and time filters
- Preview individual PLcam pixel response maps
- Choose the optimal grid centre `(xc, yc)` and grid width `map_width`

In this notebook we use `explore_grid()` in Python, which returns the same information without the interactive browser UI.

`explore_grid` signature:
```python
explore_grid(giant_h5, map_n, map_width,
             xc=None, yc=None,
             time_min=None, time_max=None,
             maxpix_min=None, maxpix_max=None,
             pix2mas=16.2,
             plcam_pixels=None,
             roi_access_key='plcam/roi_access',
             plot=True)
```
Returns a `dict` with keys: `xbins, ybins, x_mas, y_mas, xc, yc, mask, nframes_map, avg_psf_map, pixel_maps`.

In [ ]:
%%time
print("Building ROI access cache ...")
result = build_ROI_access_from_config(str(OBS_CONFIG))
print("build_ROI_access_from_config returned:", result)

roi_h5 = str(OUTPUT_DIR / "roi.h5")
if os.path.exists(roi_h5):
    print(f"roi.h5 written: {os.path.getsize(roi_h5)/1e6:.1f} MB")

alldata_h5 = str(OUTPUT_DIR / "alldata.h5")

# Preview grid coverage with plot=False, then make our own plot
grid_result = explore_grid(
    alldata_h5,
    map_n=5,
    map_width=2,
    xc=18.26,
    yc=20.50,
    pix2mas=16.2,
    plcam_pixels=[(200, 10)],
    plot=False,
)

print("\nexplore_grid result keys:", list(grid_result.keys()))
print("nframes_map:")
print(grid_result['nframes_map'])

In [ ]:
nframes_map = grid_result['nframes_map']
x_mas       = grid_result['x_mas']
y_mas       = grid_result['y_mas']

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(nframes_map, origin='lower', cmap='Blues',
               extent=[x_mas.min(), x_mas.max(), y_mas.min(), y_mas.max()])
plt.colorbar(im, ax=ax, label='N frames')

for i in range(nframes_map.shape[0]):
    for j in range(nframes_map.shape[1]):
        ax.text(x_mas[j], y_mas[i], str(nframes_map[i, j]),
                ha='center', va='center', fontsize=9, color='black')

ax.set_xlabel('x (mas)')
ax.set_ylabel('y (mas)')
ax.set_title('Frame count per grid bin')
plt.tight_layout()
plt.show()

In [ ]:
# Explore grid: centroid scatter with grid lines + pixel response map
alldata_h5 = str(OUTPUT_DIR / "alldata.h5")

grid_result = explore_grid(
    alldata_h5,
    map_n=5,
    map_width=2,
    xc=18.26,
    yc=20.50,
    pix2mas=16.2,
    plcam_pixels=[(200, 10)],
    plot=False,
)

xbins = grid_result['xbins']
ybins = grid_result['ybins']
nframes_map  = grid_result['nframes_map']
pixel_maps   = grid_result['pixel_maps']
x_mas        = grid_result['x_mas']
y_mas        = grid_result['y_mas']

with h5py.File(alldata_h5, 'r') as f:
    centroids = f['psfcam/centroids'][:]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: centroid scatter with grid lines
ax1.scatter(centroids[:, 0], centroids[:, 1], s=5, alpha=0.4, color='steelblue')
for xb in xbins:
    ax1.axvline(xb, color='red', lw=0.8, ls='--')
for yb in ybins:
    ax1.axhline(yb, color='red', lw=0.8, ls='--')
ax1.set_xlabel('PSFcam x (pixels)')
ax1.set_ylabel('PSFcam y (pixels)')
ax1.set_title('Centroid scatter with 5x5 grid overlay')
ax1.set_aspect('equal')

# Right: response map for pixel (200, 10)
key = list(pixel_maps.keys())[0]
pmap = pixel_maps[key]
im2 = ax2.imshow(pmap, origin='lower', cmap='hot',
                 extent=[x_mas.min(), x_mas.max(), y_mas.min(), y_mas.max()])
plt.colorbar(im2, ax=ax2, label='Mean PLcam counts')
ax2.set_xlabel('x (mas)')
ax2.set_ylabel('y (mas)')
ax2.set_title(f'PLcam response map — pixel {key}')

plt.tight_layout()
plt.show()

print("Tip: Open PLred/scripts/h5_viewer.html in a browser for interactive exploration of alldata.h5.")

## Step 4: Spatial Averaging → `map.h5`

Frames in `alldata.h5` are sorted into the spatial grid defined by `(map_n, map_width, xc, yc)`. All PLcam frames whose corresponding PSF centroid falls inside a given grid bin are averaged together to produce one representative detector image per bin. The output `map.h5` contains:

- `avg_PLcam` — mean PLcam frame per bin `(map_n, map_n, ny_roi, nx_roi)`
- `metadata/nframes` — number of frames in each bin `(map_n, map_n)`
- `metadata/x_mas`, `metadata/y_mas` — sky position of each bin centre in mas

Empty bins (no frames) are filled with zeros and their `nframes` entry is 0.

In [ ]:
%%time
print("Running spatial averaging ...")
average_to_h5_from_config(str(OBS_CONFIG))

map_h5 = str(OUTPUT_DIR / "map.h5")
print()
print(f"map.h5 written: {os.path.getsize(map_h5)/1e6:.2f} MB")

with h5py.File(map_h5, 'r') as f:
    avg_shape = f['avg_PLcam'].shape
    nframes   = f['metadata/nframes'][:]
    x_mas     = f['metadata/x_mas'][:]
    y_mas     = f['metadata/y_mas'][:]

print(f"  avg_PLcam shape : {avg_shape}")
print(f"  x_mas           : {x_mas}")
print(f"  y_mas           : {y_mas}")
print(f"  nframes per bin :")
print(nframes)

In [ ]:
map_h5 = str(OUTPUT_DIR / "map.h5")

with h5py.File(map_h5, 'r') as f:
    avg_PLcam = f['avg_PLcam'][:]
    nframes   = f['metadata/nframes'][:]
    x_mas     = f['metadata/x_mas'][:]
    y_mas     = f['metadata/y_mas'][:]

MAP_N = avg_PLcam.shape[0]
vmax = np.percentile(avg_PLcam[nframes > 0], 99.5)

fig, axes = plt.subplots(MAP_N, MAP_N, figsize=(MAP_N * 2.2, MAP_N * 2.2))

for iy in range(MAP_N):
    for ix in range(MAP_N):
        ax = axes[MAP_N - 1 - iy, ix]   # flip y so origin is bottom-left
        n  = nframes[iy, ix]
        if n > 0:
            img = avg_PLcam[iy, ix]
            ax.imshow(img.T, origin='lower', cmap='inferno',
                      vmin=0, vmax=vmax, aspect='auto')
            ax.set_title(f"({x_mas[ix]:.0f}, {y_mas[iy]:.0f}) mas\nn={n}",
                         fontsize=7)
        else:
            ax.set_facecolor('#cccccc')
            ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                    transform=ax.transAxes, fontsize=8, color='gray')
            ax.set_title(f"({x_mas[ix]:.0f}, {y_mas[iy]:.0f}) mas",
                         fontsize=7)
        ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Average PLcam frame per spatial bin (Step 4 output)", fontsize=13)
plt.tight_layout()
plt.show()

## Step 5a: Find Fiber Traces → `traces.npz`

Before extracting spectra we need to know precisely where each of the 38 fiber spectra falls on the detector. `make_trace_file` takes a bright flat or lamp FITS, runs:
1. `find_peaks` — detects the 38 cross-dispersion peak positions in a reference column
2. `find_traces` — tracks each fiber column-by-column and fits a polynomial trace

The result is saved as `traces.npz` and is passed to `make_trace_extractor` in Step 5b.

Run this step whenever fiber positions change (realignment, new instrument configuration). For this tutorial we also demonstrate the individual `find_peaks` / `find_traces` API so you can tune parameters interactively.

In [ ]:
map_h5 = str(OUTPUT_DIR / "map.h5")

with h5py.File(map_h5, 'r') as f:
    avg_PLcam = f['avg_PLcam'][:]
    nframes   = f['metadata/nframes'][:]

# Use the brightest populated bin as the reference image
iy_best, ix_best = np.unravel_index(np.argmax(nframes), nframes.shape)
ref_image = avg_PLcam[iy_best, ix_best]   # shape (412, 20)
print(f"Reference bin: iy={iy_best}, ix={ix_best}, n_frames={nframes[iy_best, ix_best]}")
print(f"Reference image shape: {ref_image.shape}")

# Step 1: detect peak positions
ylocs = specextract.find_peaks(ref_image, nfib=38, thres=0.045, min_dist=6, plot=False)
print(f"Detected {len(ylocs)} fiber peaks")
print("First 5 ylocs:", ylocs[:5])

# Step 2: trace each fiber
traces = specextract.find_traces(ref_image, nfib=38, ini_ys=ylocs,
                                  trace_width=4, poly_deg=2, plot=False)
print(f"Traces shape: {traces.shape}  (nfib x nx)")

# Plot: image + overlaid traces + cross-dispersion profile
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.imshow(ref_image, origin='lower', cmap='inferno', aspect='auto')
xs = np.arange(ref_image.shape[1])
for t in traces:
    ax1.plot(xs, t, color='cyan', lw=0.6, alpha=0.8)
ax1.set_title('Reference image with overlaid fiber traces')
ax1.set_xlabel('Spectral column (pixel)')
ax1.set_ylabel('Cross-dispersion (pixel)')

# Cross-dispersion profile at centre column
cx = ref_image.shape[1] // 2
profile = ref_image[:, cx]
ax2.plot(profile, np.arange(len(profile)), color='steelblue', lw=1)
for y in ylocs:
    ax2.axhline(y, color='red', lw=0.7, alpha=0.7)
ax2.set_xlabel('Signal (counts)')
ax2.set_ylabel('Cross-dispersion (pixel)')
ax2.set_title(f'Cross-dispersion profile at column {cx} (38 peaks marked)')

plt.tight_layout()
plt.show()

In [ ]:
traces_outpath = str(OUTPUT_DIR / "traces.npz")

print("Running make_trace_file ...")
result_path = specextract.make_trace_file(
    fits_path   = str(FLAT_FITS),
    nfib        = 38,
    dark_path   = str(DARK_FITS),
    xmin        = 1200,
    xmax        = 1220,
    thres       = 0.05,
    min_dist    = 6,
    trace_width = 4,
    poly_deg    = 2,
    outpath     = traces_outpath,
    plot        = False,
    verbose     = True,
)

print(f"\ntraces.npz written: {os.path.getsize(traces_outpath)/1e3:.1f} kB")

npz = np.load(traces_outpath, allow_pickle=False)
print("npz contents:")
for k in npz.files:
    print(f"  {k}: shape={npz[k].shape}, dtype={npz[k].dtype}")

## Step 5b: Extract Spectra → `couplingmap.fits`

The extractor applies the fiber traces to every spatial bin in `map.h5`. For each bin, the averaged PLcam frame is passed through `make_trace_extractor`, which sums flux within a fixed-width box centred on each traced fiber position for each spectral column. The result is a `(map_n, map_n, nfib, nwav)` array stored as a FITS file.

This is the **coupling map** — the core calibration product that maps each sky position to a set of fiber fluxes across wavelength. It is the input to `CouplingMapModel` and `PLMapFit` for image reconstruction.

In [ ]:
%%time
# The trace_file path in obs.ini already points to the traces.npz we just made.
# Confirm:
cfg_check = ConfigObj(str(OBS_CONFIG))
print("[Specextract] trace_file =", cfg_check['Specextract']['trace_file'])
print()

print("Running spectral extraction ...")
from PLred.specextract import extract_from_config
extract_from_config(str(OBS_CONFIG))

cm_path = str(OUTPUT_DIR / "couplingmap.fits")
print()
print(f"couplingmap.fits written: {os.path.getsize(cm_path)/1e6:.2f} MB")
print("Path:", cm_path)

In [ ]:
cm_path = str(OUTPUT_DIR / "couplingmap.fits")

with fits.open(cm_path) as hdl:
    hdl.info()
    print()
    primary = hdl[0]
    print("Primary HDU shape:", primary.data.shape if primary.data is not None else 'None')
    for i, hdu in enumerate(hdl):
        if hdu.data is not None:
            print(f"  Extension {i} ({hdu.name}): shape={hdu.data.shape}, dtype={hdu.data.dtype}")
        else:
            print(f"  Extension {i} ({hdu.name}): header-only")

# Try to find nframes and valid bins
map_h5 = str(OUTPUT_DIR / "map.h5")
with h5py.File(map_h5, 'r') as f:
    nframes = f['metadata/nframes'][:]

valid_bins = int((nframes > 0).sum())
print(f"\nNframes per bin:")
print(nframes)
print(f"Valid (non-empty) bins: {valid_bins} / {nframes.size}")

In [ ]:
cm_path = str(OUTPUT_DIR / "couplingmap.fits")
map_h5  = str(OUTPUT_DIR / "map.h5")

with fits.open(cm_path) as hdl:
    # find the coupling map array
    cm = None
    for hdu in hdl:
        if hdu.data is not None and hdu.data.ndim == 4:
            cm = hdu.data   # shape (map_n, map_n, nfib, nwav) or (nfib, nwav, map_n, map_n)
            print(f"Coupling map found in extension '{hdu.name}': shape={cm.shape}")
            break

if cm is None:
    print("Could not find a 4-D extension; showing primary data.")
    with fits.open(cm_path) as hdl:
        cm = hdl[0].data
    print("Primary data shape:", cm.shape)

with h5py.File(map_h5, 'r') as f:
    nframes = f['metadata/nframes'][:]
    x_mas   = f['metadata/x_mas'][:]
    y_mas   = f['metadata/y_mas'][:]

MAP_N = nframes.shape[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Panel (a): mean coupling map for fiber 0, summed over spectral axis
try:
    if cm.shape[0] == MAP_N and cm.shape[1] == MAP_N:
        # shape (map_n, map_n, nfib, nwav)
        fiber_sky = cm[:, :, 0, :].sum(axis=-1)
    else:
        # shape (nfib, nwav, map_n, map_n)
        fiber_sky = cm[0, :, :, :].sum(axis=0)
    fiber_sky_masked = np.where(nframes > 0, fiber_sky, np.nan)
    im1 = ax1.imshow(fiber_sky_masked, origin='lower', cmap='hot',
                     extent=[x_mas.min(), x_mas.max(), y_mas.min(), y_mas.max()])
    plt.colorbar(im1, ax=ax1, label='Summed coupling (counts)')
    ax1.set_xlabel('x (mas)'); ax1.set_ylabel('y (mas)')
    ax1.set_title('Fiber 0 coupling map (spectral sum)')
except Exception as e:
    ax1.text(0.5, 0.5, str(e), transform=ax1.transAxes, ha='center')

# Panel (b): spectra from most-populated bin for all fibers
try:
    iy_best, ix_best = np.unravel_index(np.argmax(nframes), nframes.shape)
    if cm.shape[0] == MAP_N and cm.shape[1] == MAP_N:
        spectra = cm[iy_best, ix_best]  # (nfib, nwav)
    else:
        spectra = cm[:, :, iy_best, ix_best]  # (nfib, nwav)
    nfib_plot, nwav = spectra.shape
    cmap_lines = plt.cm.rainbow(np.linspace(0, 1, nfib_plot))
    for i in range(nfib_plot):
        ax2.plot(spectra[i], color=cmap_lines[i], lw=0.8, alpha=0.7)
    ax2.set_xlabel('Spectral channel')
    ax2.set_ylabel('Counts')
    ax2.set_title(f'All {nfib_plot} fiber spectra — bin ({x_mas[ix_best]:.0f}, {y_mas[iy_best]:.0f}) mas')
except Exception as e:
    ax2.text(0.5, 0.5, str(e), transform=ax2.transAxes, ha='center')

plt.tight_layout()
plt.show()

## What's Next

You now have a calibrated coupling map in `tutorial_output/couplingmap.fits`. The next steps are:

1. **Load into `CouplingMapModel`** — fit a polynomial interpolation over the spatial grid:
   ```python
   from PLred.mapmodel import CouplingMapModel
   model = CouplingMapModel('tutorial_output/couplingmap.fits')
   model.fit_polynomial(deg=2)
   model.save('tutorial_output/polymodel.fits')
   ```

2. **Build the convolution matrix and run image reconstruction** — see `PLred/tutorials/step3_image_reconstruction.ipynb`.

---

## CLI Reference

All five steps read from the **same `obs.ini`**:

```bash
# Step 1: Timestamp matching  (reads [Fastcam]/[Slowcam]/[Output]/[Options])
plred-sort    obs.ini

# Step 2: Ingest              (reads [Ingest])
plred-ingest  obs.ini

# Step 3: ROI viewer cache    (reads [ROIViewer])
plred-roi     obs.ini

# Step 4: Spatial averaging   (reads [Average])
plred-average obs.ini

# Step 5a: Find fiber traces (calibration — run once per instrument config)
plred-traces data/slowcam/cropped_firstpl_15:05:17.315924371.fits \\
    --dark data/slowcam/dark.fits --nfib 38 \\
    --xmin 1200 --xmax 1220 --out tutorial_output/traces.npz

# Step 5b: Extract spectra    (reads [Specextract])
plred-extract obs.ini

# Or run all steps 2–5 with the master command:
plred-run obs.ini

# Re-run from step 4 (e.g. after tuning map parameters in obs.ini):
plred-run obs.ini --from 4
```
